In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / ".git").exists() and cwd.name == repo_name:
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Repository ready: {repo}")

## What this notebook proves

This notebook reproduces the fulfilment arithmetic behind the adopted delivery cost and topology.

It separates the volume-weighted city gig leg from the fixed-roster in-gate leg, shows the batching and SLA logic, and compares the per-gate base case with pooling as a conditional upside.

Expected headline result: **Rs17.61/order = Rs14.25 city leg + Rs3.36 in-gate leg, using 2 runners × 3 gates.**

In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import subprocess, sys

model_dir = str(Path("Model").resolve())
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

import sla as fulfilment
import fleet_mix as fleet

rows, weighted_cost = fulfilment.volume_weighted()
print(f"{'DEMAND BAND':<28} {'ORDER SHARE':>12} {'BATCH':>8} {'Rs/ORDER':>10} {'SLA MIN':>10}")
print("-" * 72)
for name, hours, multiplier, share, rate, batch, cost, sla_min, city, in_gate in rows:
    print(f"{name:<28} {share:>11.1%} {batch:>8} {cost:>10.1f} {sla_min:>10.1f}")

runners, alpha, in_gate_cost, gates = fleet.plan_roster()
legs = fulfilment.cost_legs()
print(f"\nVolume-weighted fulfilment cost = Rs{weighted_cost:.2f}/order")
print(f"City leg Rs{legs['city']:.2f} + in-gate leg Rs{legs['in_gate']:.2f}")
print(f"Topology: {gates} gates | {runners} runners | rostered share {alpha:.1%}")

full_parts = []
for script in ("Model/sla.py", "Model/fleet_mix.py"):
    full_result = subprocess.run([sys.executable, script], text=True, capture_output=True)
    full_parts.append(f"### {script}\n{full_result.stdout}" + (("\nSTDERR\n" + full_result.stderr) if full_result.stderr.strip() else ""))
    full_result.check_returncode()
full_text = "\n\n".join(full_parts)
display(HTML(
    "<details style='margin-top:12px'><summary><b>Show full fulfilment and fleet reports</b></summary>"
    f"<pre style='white-space:pre-wrap'>{escape(full_text)}</pre></details>"
))

## How to read the result

Use the SLA output for the service-time identity and the fleet output for roster, gig, pooling, and utilisation economics.

The city leg scales with volume; the in-gate leg is roster cost divided by daily volume. Pooling is not the base case unless the pilot establishes cross-gate movement and reprices repositioning.